In [8]:
from delphi_epidata import Epidata
import pandas as pd

START_YEAR = 2010
END_YEAR = 2025

In [ ]:
def fetch_fluview(regions, start_year, end_year):
    rows = []
    for y in range(start_year, end_year + 1):
        ew_start = y * 100 + 1
        ew_end   = y * 100 + 53
        res = Epidata.fluview(regions, [Epidata.range(ew_start, ew_end)])
        if res["result"] not in (1, 2):
            raise RuntimeError(f"{y}: {res['message']}")
        rows.extend(res.get("epidata", []))
    return pd.DataFrame(rows)

def region_type(label: str) -> str:
    if label == "nat":
        return "National"
    if label.startswith("hhs"):
        return "HHS Region"
    if label.startswith("cen"):
        return "Census Region"
    # two-letter states + dc etc.
    if len(label) == 2:
        return "State"
    return "Other"

def format_columns(df: pd.DataFrame) -> pd.DataFrame:
    # YEAR / WEEK from epiweek
    df["YEAR"] = (df["epiweek"] // 100).astype(int)
    df["WEEK"] = (df["epiweek"] % 100).astype(int)

    # Region fields
    df["REGION"] = df["region"]
    df["REGION TYPE"] = df["region"].map(region_type)

    # ILI percents
    df["% WEIGHTED ILI"] = df["wili"]
    df["%UNWEIGHTED ILI"] = df["ili"]

    # DELPHI agegroups
    # num_age_0 = 0–4
    # num_age_1 = 5–24
    # num_age_2 = 25–64 (older reporting; often null now)
    # num_age_3 = 25–49 (newer reporting)
    # num_age_4 = 50–64 (newer reporting)
    # num_age_5 = 65+
    # One of {num_age_2} or {num_age_3,num_age_4} may be null depending on year. :contentReference[oaicite:2]{index=2}

    df["AGE 0-4"]   = df["num_age_0"]
    df["AGE 5-24"]  = df["num_age_1"]
    df["AGE 25-49"] = df["num_age_3"]
    df["AGE 50-64"] = df["num_age_4"]
    df["AGE 65"]    = df["num_age_5"]

    # if combined bin is present we use 25–64, otherwise sum split bins
    df["AGE 25-64"] = df["num_age_2"]
    mask_missing_2564 = df["AGE 25-64"].isna()
    df.loc[mask_missing_2564, "AGE 25-64"] = (
        df.loc[mask_missing_2564, "AGE 25-49"].fillna(0)
        + df.loc[mask_missing_2564, "AGE 50-64"].fillna(0)
    )

    # Totals / denominators
    df["ILITOTAL"] = df["num_ili"]
    df["NUM. OF PROVIDERS"] = df["num_providers"]
    df["TOTAL PATIENTS"] = df["num_patients"]

    cols = [
        "REGION TYPE","REGION","YEAR","WEEK",
        "% WEIGHTED ILI","%UNWEIGHTED ILI",
        "AGE 0-4","AGE 25-49","AGE 25-64","AGE 5-24","AGE 50-64","AGE 65",
        "ILITOTAL","NUM. OF PROVIDERS","TOTAL PATIENTS"
    ]
    return df[cols].sort_values(["REGION TYPE","REGION","YEAR","WEEK"])


# HHS regions 
hhs_regions = [f"hhs{i}" for i in range(1, 11)]
raw_hhs = fetch_fluview(hhs_regions, START_YEAR, END_YEAR)
hhs_table = format_columns(raw_hhs)
hhs_table.to_csv(f"data/hhs_ilinet_{START_YEAR}_{END_YEAR}.csv", index=False)

# States 
states = [
    "al","ak","az","ar","ca","co","ct","de","fl","ga",
    "hi","id","il","in","ia","ks","ky","la","me","md",
    "ma","mi","mn","ms","mo","mt","ne","nv","nh","nj",
    "nm","ny","nc","nd","oh","ok","or","pa","ri","sc",
    "sd","tn","tx","ut","vt","va","wa","wv","wi","wy",
    "dc"
]
raw_states = fetch_fluview(states, START_YEAR, END_YEAR)
states_table = format_columns(raw_states)
states_table.to_csv(f"data/states_ilinet_{START_YEAR}_{END_YEAR}.csv", index=False)